# 02. Cell re-segmentation with Proseg

**Why:** in the core workflow (Step 6), ~10% of cells were flagged `mixed`: their 10x cell outlines captured transcripts from neighbouring cells (segmentation spillover). Whole populations suffered: CD4 T cells were absorbed into a mixed T/endothelial cluster and dropped out of every spatial statistic, and tumour-contact T cells carried epithelial transcripts (Step 9b control row).

**What:** re-segment the tissue from the transcripts themselves with **Proseg** (Jones et al., *Nature Methods* 2025), a probabilistic method that assigns each transcript to the cell that most plausibly produced it, using the 10x nuclei as a starting point.

**How the comparison stays fair:** this notebook only produces a new segmentation and its count matrix. The core workflow then re-runs Steps 4-10 on it with identical code (a `SEGMENTATION` switch), so any difference comes from segmentation alone.

| Part | Content |
|---|---|
| A | Baseline: how good is the 10x segmentation? (metrics a new segmentation must beat) |
| B | Prepare inputs and run Proseg |
| C | Load the Proseg output and compare with the baseline |
| D | Export for the core workflow |


## Setup

Shared paths, constants and helpers come from `xenium_utils.py` at the repository root.

In [ ]:
import sys
from pathlib import Path

# make the repository root (where xenium_utils.py lives) importable from notebooks/
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "xenium_utils.py").exists())
sys.path.insert(0, str(root))
import xenium_utils as xu

xu.check_kernel()

import warnings

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
OUT_DIR = xu.output_dir("resegmentation")
print(f"Outputs go to {xu.display_path(OUT_DIR)}")

## A. Baseline: quality of the 10x segmentation

Four numbers any re-segmentation should improve, or at least not worsen:
1. **Transcripts assigned to cells**: more is better, as long as cells stay plausible.
2. **Cells flagged `mixed`** (Step 6): direct measure of spillover.
3. **Lineage-impossible co-expression**: % of T cells with epithelial transcripts (EPCAM or MALL) and % of tumour cells with T-cell transcripts (CD3E or TRAC). In a clean segmentation both approach the background rate.
4. **Recovered populations**: whether CD4 T cells form a clean cluster.

In [ ]:
# 1. Transcript assignment (gene transcripts, qv >= 20)
tx = pq.read_table(xu.OUTS_DIR / "transcripts.parquet", columns=["feature_name", "qv", "cell_id"])
is_gene = ~pc.match_substring_regex(tx["feature_name"], "^(NegControl|Unassigned|Deprecated)").to_numpy(zero_copy_only=False)
q20 = (tx["qv"].to_numpy() >= 20) & is_gene
assigned = pc.not_equal(tx["cell_id"], "UNASSIGNED").to_numpy(zero_copy_only=False)
frac_assigned = (q20 & assigned).sum() / q20.sum()
del tx

# 2-3. Spillover measures from the Step 6 annotation (all QC-passing cells)
ann = xu.load_step("annotated")
counts = ann.layers["counts"].tocsc()


def positive(genes):
    idx = [ann.var_names.get_loc(g) for g in genes]
    return np.asarray((counts[:, idx] > 0).sum(axis=1)).ravel() > 0


lineage = ann.obs["lineage"].astype(str).to_numpy()
is_t = lineage == "T / NK"
is_tumour = ann.obs["cell_type"].isin(xu.TUMOUR_TYPES).to_numpy()
baseline = pd.Series({
    "gene transcripts (qv>=20) assigned to a cell": f"{frac_assigned:.1%}",
    "cells after QC": f"{ann.n_obs:,}",
    "cells flagged 'mixed' (Step 6)": f"{(ann.obs['annotation_confidence'] == 'mixed').mean():.1%}",
    "T / NK cells with EPCAM or MALL": f"{positive(['EPCAM', 'MALL'])[is_t].mean():.1%}",
    "tumour cells with CD3E or TRAC": f"{positive(['CD3E', 'TRAC'])[is_tumour].mean():.1%}",
    "CD4 T cells, high confidence": f"{((ann.obs['cell_type'] == 'CD4 T') & (ann.obs['annotation_confidence'] == 'high')).sum():,}",
}, name="10x segmentation (baseline)")
display(baseline.to_frame())
baseline.to_csv(OUT_DIR / "baseline_10x_segmentation.csv")

## B. Prepare inputs and run Proseg

*To do.* Proseg is a command-line tool written in Rust. It reads `transcripts.parquet` directly (it has a built-in Xenium mode) and writes per-cell counts, cell metadata and polygons.

Requires a one-time install of the Rust toolchain and `cargo install proseg`.

## C. Compare with the baseline

*To do:* same four metrics as part A, plus cell size distributions and a side-by-side map of outlines on the DAPI image.

## D. Export for the core workflow

*To do:* write the Proseg counts as a Xenium-like output the core workflow can read via its `SEGMENTATION` switch.